In [0]:
import time
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady


def wait_for_endpoint_ready(name, timeout_s=900, poll_s=15):
    """
    Wait for endpoint to be ready.
    
    Args:
        name: Endpoint name
        timeout_s: Maximum wait time in seconds (default: 15 minutes)
        poll_s: Polling interval in seconds
    """
    deadline = time.time() + timeout_s
    failure_states = {
        EndpointStateConfigUpdate.UPDATE_FAILED,
        EndpointStateConfigUpdate.UPDATE_CANCELED,
    }
    
    print(f"🔄 Waiting for endpoint '{name}' to be ready...")
    print(f"   Timeout: {timeout_s}s ({timeout_s // 60} minutes)")
    print(f"   Polling every {poll_s}s\n")
    
    elapsed = 0
    while time.time() < deadline:
        state = w.serving_endpoints.get(name).state
        
        if (
            state.ready == EndpointStateReady.READY
            and state.config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            print(f"\n✅ Endpoint '{name}' is READY!")
            print(f"   Total deployment time: {elapsed}s ({elapsed // 60} min {elapsed % 60} sec)")
            return
        
        if state.config_update in failure_states:
            print(f"\n❌ Deployment FAILED")
            print(f"   State: {state.config_update.value}")
            raise RuntimeError(
                f"{name} deployment failed with config update state {state.config_update.value}"
            )
        
        # Progress update every minute
        if elapsed % 60 == 0:
            print(f"   ⏳ Still deploying... ({elapsed}s elapsed, state: {state.config_update.value})")
        
        time.sleep(poll_s)
        elapsed += poll_s
    
    raise TimeoutError(f"❌ Endpoint '{name}' not ready after {timeout_s}s")


# Wait for readiness
try:
    wait_for_endpoint_ready(endpoint_name)
    
    # Display endpoint details
    endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"\n🎉 DEPLOYMENT SUCCESSFUL!")
    print("=" * 60)
    print(f"   Endpoint Name: {endpoint_name}")
    print(f"   State: {endpoint.state.ready.value}")
    print(f"   Model: {registered_model_name} (v{model_version})")
    print(f"\n🔗 Endpoint URL:")
    print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
    print("\n📡 Ready to serve predictions!")
    print("=" * 60)
    
except (TimeoutError, RuntimeError) as e:
    print(f"\n❌ Deployment Error: {str(e)}")
    print(f"\n👉 Check endpoint status at:")
    print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
    raise

In [0]:
# Test the deployed endpoint
print("🧪 Testing deployed endpoint...\n")

# Prepare test input - match the model signature format
import pandas as pd

test_questions = pd.DataFrame({
    'question': [
        "What are the main customer concerns about product delivery?",
        "What positive feedback has been received about the product demo?"
    ]
})

try:
    # Query the endpoint with DataFrame input
    response = w.serving_endpoints.query(
        name=endpoint_name,
        dataframe_records=test_questions.to_dict(orient='records')
    )
    
    print("✅ Endpoint Test Successful!\n")
    print("="*80)
    print("TEST RESULTS")
    print("="*80)
    
    # Display predictions
    predictions = response.predictions
    for i, pred in enumerate(predictions, 1):
        print(f"\n💬 Question {i}:")
        print(pred.get('question', 'N/A'))
        print(f"\n📝 Answer {i}:")
        print(pred.get('answer', 'N/A'))
        print("\n" + "-"*80)
    
    print("\n✨ Endpoint is serving predictions correctly!")
    
except Exception as e:
    print(f"❌ Endpoint test failed: {str(e)}")
    print("\n👉 Troubleshooting steps:")
    print("   1. Verify endpoint is in READY state")
    print("   2. Check endpoint logs for errors")
    print("   3. Validate model dependencies")
    print(f"   4. Check endpoint at: https://{w.config.host}/ml/endpoints/{endpoint_name}")
    raise